# Option B: full policy workflow

A deliberately simple agentic pipeline for Bachelor admissions screening, built to be compared
side by side against the rule-based system in `app/`, which is a facts extractor followed by a
deterministic rules engine.

**Thesis under test.** Can "one agent reads the policy, a second agent reads the applicant's
documents and decides" match the extractor and rules engine architecture on accuracy and
stability?

```
Leitfaden (policy, English .md) ──▶ [ policy_analyst ] ──criteria──▶ [ decision_maker ] ──▶ decision
applicant PDFs (digital) ───────────────────────────────────────────────────┘
                                              optional:  decision ──▶ [ critic ] ──▶ final decision
```

## Ground rules, agreed before the build

| Rule | Why |
| --- | --- |
| Same model as the rule-based system, set by `ADMISSIONS_OPENAI_MODEL` | One variable at a time, meaning architecture rather than model quality |
| The policy is read fresh **once per applicant run**, with no caching | That is the naive version being tested |
| The policy input is the official handbook itself, the English `.md` conversion, **not** the curated audio-criteria doc | Otherwise the prototype free-rides on the rule extractor's work |
| The output shares the 5-value `application_status` vocabulary, and the rule taxonomy (`GERMAN_ABITUR`, and so on) and reason codes are **withheld** | The status enum is answer vocabulary. The rule taxonomy is the distilled policy knowledge the prototype has to earn itself |
| Scope is **academic entrance qualification only**, so authentication, translations, CVs, and insurance are out of bounds | Matches the rules engine's `ACADEMIC_ACCESS_ONLY` scope. A mismatched scope makes disagreements uninterpretable |
| Zero imports from `app/` | Clean-room build |

Every run is persisted. Per-call telemetry goes to `runs/ledger.jsonl`, one decision record per
run to `runs/prototype-results.jsonl`, the rule-based baseline to `baseline/<persona>/`, and the
evaluation to `runs/eval-summary.json`, `runs/comparison.csv`, and `runs/disagreements.md`. Full
raw responses are written to `runs/raw/` on a run, and the committed copy was removed to keep the
repository small.


## 0. Configuration

Everything tunable lives here. The notebook reads from the repository root, meaning the policy,
the sample PDFs, and `.env`, and writes **only** inside this experiment folder.


In [ ]:
import base64
import hashlib
import json
import os
import threading
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal, Optional, TypedDict

from dotenv import load_dotenv

# --- Locations -----------------------------------------------------------------
EXPERIMENT_DIR = Path.cwd()
REPO_ROOT = EXPERIMENT_DIR.parent.parent
LEITFADEN_MD = REPO_ROOT / "case-study" / "IU-FS-LF-Leitfaden-Hochschulzugangsberechtigung-Stand-Januar2025.md"
SAMPLES_DIR = REPO_ROOT / "samples" / "filled-documents"
RUNS_DIR = EXPERIMENT_DIR / "runs"
RAW_DIR = RUNS_DIR / "raw"
RESULTS_PATH = RUNS_DIR / "prototype-results.jsonl"
LEDGER_PATH = RUNS_DIR / "ledger.jsonl"
BASELINE_DIR = EXPERIMENT_DIR / "baseline"   # written by baseline.sh (fresh `admissions screen` runs)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# --- Experiment knobs ------------------------------------------------------------
MODEL = os.getenv("TWO_AGENTS_MODEL", os.getenv("ADMISSIONS_OPENAI_MODEL", "gpt-5.4-mini"))
# Keep this the same as the rule-based system and options C/D, or the comparison measures two variables.
MAX_OUTPUT_TOKENS = 16_000
ENABLE_CRITIC = True            # toggle the third (verifier) agent, then re-run the graph cells
N_REPEATS = 3                    # repeat runs per persona, for the stability (flip-rate) metric

# Batch collection is idempotent: completed (persona, repeat) runs are cached in
# runs/prototype-results.jsonl and never re-run, so RUN_BATCH=True is safe to re-execute.
RUN_BATCH = True
BATCH_WORKERS = 4                # concurrent pipeline runs (each policy read is ~95k input tokens)
BATCH_TIME_BUDGET_S = 400        # stop submitting new runs after this; re-execute to continue

# The program the synthetic applicants apply to (mirrors config/programs.yaml: BACHELOR / COMPUTER_SCIENCE).
PROGRAM_CONTEXT = {
    "program": "Bachelors Study Program",
}

# --- Secrets & tracing -----------------------------------------------------------
load_dotenv(REPO_ROOT / ".env")
os.environ["LANGSMITH_PROJECT"] = os.getenv("TWO_AGENTS_LANGSMITH_PROJECT", "auto-admissions-prototype")   # separate from the root app's project
os.environ.setdefault("LANGSMITH_TRACING", "true")

assert LEITFADEN_MD.exists(), f"{LEITFADEN_MD}\nPut the English markdown handbook at this path. It is IU material and is not in the repository: see case-study/README.md."
assert SAMPLES_DIR.exists(), SAMPLES_DIR
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing - check the repo root .env"
print(f"model={MODEL}  critic={ENABLE_CRITIC}  repeats={N_REPEATS}  batch={RUN_BATCH}")
print(f"policy: {LEITFADEN_MD.name} ({LEITFADEN_MD.stat().st_size / 1024:.0f} KB)")
print(f"personas: {sum(1 for p in SAMPLES_DIR.iterdir() if p.is_dir())}")

## 1. Observability

Three layers, all cheap.

1. **LangSmith tracing.** The OpenAI client is wrapped with `wrap_openai`, so every model call
   lands in the `auto-admissions-prototype` project with its prompt, response, tokens, and
   latency. Content is traced in full, which is the same stance as D8 in the root design
   document.
2. **Run ledger.** `runs/ledger.jsonl` holds one line per model call, with the node, tokens,
   latency, and prompt hash. The cost and latency comparison reads it.
3. **Raw responses.** Every full API response is saved under `runs/raw/`, so a disagreement can
   be audited later without re-running anything.


In [2]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai

client = wrap_openai(OpenAI(max_retries=2))
_JSONL_LOCK = threading.Lock()   # batch runs append from worker threads


def _append_jsonl(path: Path, record: dict) -> None:
    with _JSONL_LOCK:
        with path.open("a") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def call_agent(*, node: str, run_id: str, instructions: str, content: list, schema):
    """One structured-output model call, fully observed.

    Returns the parsed pydantic object; raises if the model returned no parseable
    output (refusal / truncation) so a failed call is never silently treated as data.
    """
    started = time.perf_counter()
    response = client.responses.parse(
        model=MODEL,
        instructions=instructions,
        input=[{"role": "user", "content": content}],
        text_format=schema,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        store=False,
    )
    duration_ms = round((time.perf_counter() - started) * 1000)

    raw_path = RAW_DIR / f"{run_id}--{node}.json"
    raw_path.write_text(response.model_dump_json(indent=2))
    usage = response.usage
    _append_jsonl(LEDGER_PATH, {
        "ts": datetime.now(timezone.utc).isoformat(),
        "run_id": run_id,
        "node": node,
        "model": MODEL,
        "status": response.status,
        "input_tokens": usage.input_tokens if usage else None,
        "output_tokens": usage.output_tokens if usage else None,
        "duration_ms": duration_ms,
        "instructions_sha256": hashlib.sha256(instructions.encode()).hexdigest()[:16],
        "raw": str(raw_path.relative_to(EXPERIMENT_DIR)),
    })

    if response.output_parsed is None:
        raise RuntimeError(f"{node} produced no parsed output (status={response.status}) - see {raw_path}")
    return response.output_parsed

## 2. Output contracts

The line between what the prototype is given and what it has to earn runs through this cell.

- `application_status` uses the same 5 values as the rules engine's `ApplicationStatus`. That is
  answer vocabulary, and sharing it is what makes an exact status-level comparison possible.
- The rule taxonomy (`GERMAN_ABITUR`, `FACHGEBUNDENE_HOCHSCHULREIFE`, and the rest) and the
  reason codes are **deliberately absent**. They are the distilled output of the rule extractor,
  meaning the answer to which legal routes lead into a Bachelor program. The policy analyst has
  to rediscover them from the handbook, and whether it does is itself a finding.
- Per-criterion verdicts use a generic 4-value vocabulary of `FULFILLED`, `NOT_FULFILLED`,
  `UNCLEAR`, and `NOT_RELEVANT`, which gives structure without leaking policy knowledge.


In [3]:
from pydantic import BaseModel, Field

ApplicationStatus = Literal[
    "ELIGIBLE",
    "CONDITIONALLY_ELIGIBLE",
    "INELIGIBLE",
    "MISSING_INFORMATION",
    "MANUAL_REVIEW",
]


# --- Agent 1: what the policy analyst extracts from the handbook -----------------
class ExtractedCriterion(BaseModel):
    criterion_id: str = Field(description="Short stable slug the analyst chooses itself, e.g. 'abitur-direct'")
    name: str
    summary: str = Field(description="The condition in the analyst's own words, with exact thresholds")
    evidence_expected: str = Field(description="Which documents or facts prove or refute this criterion")
    decision_guidance: str = Field(description="How fulfillment/failure should influence the admission decision, incl. any conditional-admission mechanism")


class PolicyCriteria(BaseModel):
    policy_title: str
    scope_notes: str = Field(description="What the analyst treated as in/out of scope and why")
    criteria: list[ExtractedCriterion]


# --- Agent 2: the decision -------------------------------------------------------
class CriterionAssessment(BaseModel):
    criterion_id: str = Field(description="Must reference a criterion_id from the extracted criteria")
    verdict: Literal["FULFILLED", "NOT_FULFILLED", "UNCLEAR", "NOT_RELEVANT"]
    reasoning: str
    evidence: str = Field(description="Short quotation or concrete reference from the applicant's documents; empty string if none")


class AdmissionDecision(BaseModel):
    application_status: ApplicationStatus
    rationale: str = Field(description="The decisive chain of reasoning, in plain language")
    conditions: list[str] = Field(description="Conditions attached to a CONDITIONALLY_ELIGIBLE outcome; else empty")
    missing_information: list[str] = Field(description="Evidence that is absent/incomplete/unreadable; else empty")
    manual_review_reasons: list[str] = Field(description="Why a human must look at this; else empty")
    criteria_assessments: list[CriterionAssessment]


# --- Agent 3 (optional): the critic ----------------------------------------------
class CriticReview(BaseModel):
    verdict_agrees: bool
    critique: str
    revised_decision: Optional[AdmissionDecision] = Field(
        description="Full corrected decision when verdict_agrees is false; null otherwise"
    )

## 3. Prompts

Both prompts pin the scope, which is academic entrance qualification only, and the decision
semantics. What they leave out matters as much. There are no route names, no thresholds, and no
German credential vocabulary, so all of that has to come out of the handbook through agent 1.


In [4]:
POLICY_ANALYST_INSTRUCTIONS = """\
You are an admissions policy analyst. You receive the full text of a university's \
official handbook on university entrance qualification (translated to English).

Extract the complete set of criteria that govern ACADEMIC entrance qualification for \
Bachelor (undergraduate) study: every distinct route by which an applicant can qualify, \
and for each route its conditions, thresholds, and required evidence.

Rules:
- Academic entrance qualification ONLY. Ignore certified-copy/authentication \
requirements, translations, CVs, identity documents, health insurance, enrollment \
paperwork, and tuition or administrative matters.
- Use your own terminology and give each criterion a short stable criterion_id slug.
- Capture quantitative conditions exactly (durations, levels, hour counts, scopes).
- In decision_guidance, state how fulfillment or failure should influence the admission \
decision, including any conditional-admission mechanisms the handbook describes.
- Be exhaustive. It is better to include a rarely used route than to drop one.\
"""

DECISION_MAKER_INSTRUCTIONS = """\
You are an admissions decision agent. You receive (1) admission criteria extracted from \
the university's official policy, (2) the program the applicant applies to, and (3) the \
applicant's uploaded documents as PDFs.

Decide the applicant's ACADEMIC entrance qualification status. Choose exactly one:
- ELIGIBLE: the documents prove a qualifying route with no outstanding conditions.
- CONDITIONALLY_ELIGIBLE: a route applies only together with additional conditions the \
policy imposes; record them in `conditions`.
- INELIGIBLE: the documents affirmatively prove that no route is satisfied.
- MISSING_INFORMATION: the decision needs evidence that is absent, incomplete, or \
unreadable; list it in `missing_information`.
- MANUAL_REVIEW: the case cannot be decided confidently - borderline thresholds, unclear \
applicability, conflicting evidence, or situations the criteria do not clearly cover; \
explain in `manual_review_reasons`.

Rules:
- Judge ONLY academic entrance qualification. Gaps in authentication, translations, CVs, \
or insurance are out of scope and must not affect the status.
- Use only the supplied documents as evidence. Never assume facts that are not evidenced.
- Absence of evidence is MISSING_INFORMATION or MANUAL_REVIEW, never INELIGIBLE. \
INELIGIBLE requires positive proof of failure.
- Assess every criterion that could plausibly apply and record a per-criterion verdict \
with a short quotation or concrete document reference as evidence.\
"""

CRITIC_INSTRUCTIONS = """\
You are an adversarial reviewer of admission decisions. You receive the extracted policy \
criteria, the applicant's documents, and a draft decision.

Actively try to refute the draft: misread evidence, misapplied or overlooked criteria, a \
status that does not follow from the assessments, scope violations. Apply the same \
decision rules the decision agent was given (academic scope only; absence of evidence is \
never INELIGIBLE).

If the draft survives your attack, set verdict_agrees=true and revised_decision=null. \
If it is flawed, set verdict_agrees=false and provide the full corrected decision.\
"""

## 4. Agents and graph

Two LangGraph nodes, each a single structured output call, with no tools and no loops. The
optional `critic` node is wired in only when `ENABLE_CRITIC` is set, so the critic's effect on
agreement can be measured as a toggle.

> **Documented alternative, root `docs/technical-design-document.md` D1.** Tool-using agents that can re-query the
> policy mid-reasoning were considered and deliberately excluded here, because they change two
> variables at once and add nondeterminism that would pollute the flip-rate metric. Options C and
> D in `../option-cd-agentic-rag/` are that design, built separately.


In [5]:
from langgraph.graph import END, START, StateGraph

LEITFADEN_TEXT = LEITFADEN_MD.read_text()


class PipelineState(TypedDict, total=False):
    persona: str
    run_id: str
    repeat: int
    pdf_paths: list[Path]
    criteria: PolicyCriteria
    decision: AdmissionDecision
    critic: CriticReview
    final_decision: AdmissionDecision


def _pdf_block(path: Path) -> dict:
    data = base64.b64encode(path.read_bytes()).decode()
    return {"type": "input_file", "filename": path.name,
            "file_data": f"data:application/pdf;base64,{data}"}


def policy_analyst(state: PipelineState) -> PipelineState:
    criteria = call_agent(
        node="policy_analyst",
        run_id=state["run_id"],
        instructions=POLICY_ANALYST_INSTRUCTIONS,
        content=[{"type": "input_text", "text": f"Handbook text:\n\n{LEITFADEN_TEXT}"}],
        schema=PolicyCriteria,
    )
    return {"criteria": criteria}


def decision_maker(state: PipelineState) -> PipelineState:
    preamble = (
        f"Program context:\n{json.dumps(PROGRAM_CONTEXT, indent=2)}\n\n"
        f"Admission criteria (extracted from the official policy):\n"
        f"{state['criteria'].model_dump_json(indent=2)}\n\n"
        f"The applicant's documents follow ({len(state['pdf_paths'])} PDFs)."
    )
    content = [{"type": "input_text", "text": preamble}]
    content += [_pdf_block(p) for p in state["pdf_paths"]]
    decision = call_agent(
        node="decision_maker",
        run_id=state["run_id"],
        instructions=DECISION_MAKER_INSTRUCTIONS,
        content=content,
        schema=AdmissionDecision,
    )
    return {"decision": decision, "final_decision": decision}


def critic(state: PipelineState) -> PipelineState:
    preamble = (
        f"Program context:\n{json.dumps(PROGRAM_CONTEXT, indent=2)}\n\n"
        f"Admission criteria:\n{state['criteria'].model_dump_json(indent=2)}\n\n"
        f"Draft decision under review:\n{state['decision'].model_dump_json(indent=2)}\n\n"
        f"The applicant's documents follow ({len(state['pdf_paths'])} PDFs)."
    )
    content = [{"type": "input_text", "text": preamble}]
    content += [_pdf_block(p) for p in state["pdf_paths"]]
    review = call_agent(
        node="critic",
        run_id=state["run_id"],
        instructions=CRITIC_INSTRUCTIONS,
        content=content,
        schema=CriticReview,
    )
    final = state["decision"]
    if not review.verdict_agrees and review.revised_decision is not None:
        final = review.revised_decision
    return {"critic": review, "final_decision": final}


builder = StateGraph(PipelineState)
builder.add_node("policy_analyst", policy_analyst)
builder.add_node("decision_maker", decision_maker)
builder.add_edge(START, "policy_analyst")
builder.add_edge("policy_analyst", "decision_maker")
if ENABLE_CRITIC:
    builder.add_node("critic", critic)
    builder.add_edge("decision_maker", "critic")
    builder.add_edge("critic", END)
else:
    builder.add_edge("decision_maker", END)
graph = builder.compile()

print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	policy_analyst(policy_analyst)
	decision_maker(decision_maker)
	__end__([<p>__end__</p>]):::last
	__start__ --> policy_analyst;
	policy_analyst --> decision_maker;
	decision_maker --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. Run one applicant

`run_pipeline` is the unit everything else is built from. One persona, one repeat index, one full
policy read and decision, and one appended record in `runs/prototype-results.jsonl`.

**Idempotent by design.** A persona and repeat pair that already has a record for the current
critic setting returns the cached record instead of spending tokens. Pass `force=True` to re-run
deliberately.

The smoke persona is **felix-brandt**, which is tuple 1 in
`../../samples/blank-documents/eval-tuples.md`, expected `ELIGIBLE` on Abitur direct access.


In [6]:
def digital_pdfs(persona: str) -> list[Path]:
    """The digital variants only - scan robustness was measured separately and is not the variable here."""
    folder = SAMPLES_DIR / persona
    return sorted(p for p in folder.glob("*.pdf") if not p.name.endswith("-scan.pdf"))


def _load_records() -> dict:
    """All persisted pipeline records for the current critic setting, keyed by (persona, repeat)."""
    records = {}
    if RESULTS_PATH.exists():
        for line in RESULTS_PATH.open():
            r = json.loads(line)
            if r["critic_enabled"] == ENABLE_CRITIC:
                records[(r["persona"], r["repeat"])] = r
    return records


def run_pipeline(persona: str, repeat: int = 0, force: bool = False) -> dict:
    if not force:
        cached = _load_records().get((persona, repeat))
        if cached is not None:
            return cached

    run_id = f"{persona}-r{repeat}-{uuid.uuid4().hex[:8]}"
    pdfs = digital_pdfs(persona)
    assert pdfs, f"no digital PDFs for {persona}"
    state: PipelineState = graph.invoke(
        {"persona": persona, "run_id": run_id, "repeat": repeat, "pdf_paths": pdfs},
        config={"run_name": run_id},
    )
    final = state["final_decision"]
    record = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "run_id": run_id,
        "persona": persona,
        "repeat": repeat,
        "model": MODEL,
        "critic_enabled": ENABLE_CRITIC,
        "pdfs": [p.name for p in pdfs],
        "application_status": final.application_status,
        "criteria_extracted": len(state["criteria"].criteria),
        "critic_agreed": state["critic"].verdict_agrees if "critic" in state else None,
        "decision": final.model_dump(),
    }
    _append_jsonl(RESULTS_PATH, record)
    return record


def show(record: dict) -> None:
    d = record["decision"]
    print(f"persona: {record['persona']}   run: {record['run_id']}")
    print(f"status:  {d['application_status']}")
    print(f"rationale: {d['rationale']}\n")
    for a in d["criteria_assessments"]:
        print(f"  [{a['verdict']:>13}] {a['criterion_id']}: {a['reasoning'][:110]}")
    if d["conditions"]:
        print(f"conditions: {d['conditions']}")
    if d["missing_information"]:
        print(f"missing: {d['missing_information']}")
    if d["manual_review_reasons"]:
        print(f"manual review: {d['manual_review_reasons']}")
    print(f"\n(policy analyst extracted {record['criteria_extracted']} criteria)")


show(run_pipeline("felix-brandt"))

persona: felix-brandt   run: felix-brandt-r0-e73d5080
status:  ELIGIBLE
rationale: The applicant has proven a German Allgemeine Hochschulreife (Abitur) issued by Lessing-Gymnasium, Düsseldorf. This is a general German higher-education entrance qualification and provides direct access to the B.Sc. Computer Science without subject restriction or trial-study/entrance-examination conditions. The program is not among the policy's restricted, design-aptitude, or engineering-internship programs.

  [    FULFILLED] standard-german-hzb: The submitted certificate is expressly a Zeugnis der Allgemeinen Hochschulreife from a school in Düsseldorf, G
  [ NOT_RELEVANT] fachhochschulreife-general: The applicant relies on a general Abitur, not a Fachhochschulreife.
  [ NOT_RELEVANT] fachhochschulreife-subject-match: No subject-restricted Fachhochschulreife is presented; the documented Abitur is a general HZB.
  [ NOT_RELEVANT] german-language-proficiency: This criterion applies to foreign applicants. T

## 6. Batch data collection

Every persona times `N_REPEATS`, in parallel across `BATCH_WORKERS`, and resumable. Completed runs
come from the cache, and the cell stops submitting new work after `BATCH_TIME_BUDGET_S` so a
notebook execution never hangs. Re-execute until `pending` reaches 0.


In [7]:
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

PERSONAS = sorted(p.name for p in SAMPLES_DIR.iterdir() if p.is_dir())
_done = _load_records()
pending = [(p, r) for r in range(N_REPEATS) for p in PERSONAS if (p, r) not in _done]
print(f"{len(PERSONAS)} personas x {N_REPEATS} repeats: {len(_done)} cached, {len(pending)} pending")

if RUN_BATCH and pending:
    started = time.monotonic()
    queue = list(pending)
    failures = []
    completed = 0
    with ThreadPoolExecutor(max_workers=BATCH_WORKERS) as pool:
        futures = {}

        def submit_next():
            if queue and time.monotonic() - started < BATCH_TIME_BUDGET_S:
                p, r = queue.pop(0)
                futures[pool.submit(run_pipeline, p, r)] = (p, r)

        for _ in range(BATCH_WORKERS):
            submit_next()
        while futures:
            finished, _ = wait(futures, return_when=FIRST_COMPLETED)
            for fut in finished:
                p, r = futures.pop(fut)
                try:
                    rec = fut.result()
                    completed += 1
                    print(f"r{r} {p}: {rec['application_status']}")
                except Exception as exc:
                    failures.append((r, p, repr(exc)))
                    print(f"r{r} {p}: FAILED - {exc!r}")
                submit_next()

    print(f"\ncompleted {completed}, failed {len(failures)}, still pending {len(queue)} "
          f"({time.monotonic() - started:.0f}s elapsed)")
    if queue:
        print("time budget reached - re-execute this cell/notebook to continue; completed runs are cached.")
    for f in failures:
        print("  FAILED:", f)

16 personas x 3 repeats: 42 cached, 6 pending
r2 sofia-lorenz: ELIGIBLE
r2 stefan-brenner: MISSING_INFORMATION
r2 melina-sturm: MISSING_INFORMATION
r2 oemer-yilmaz: MANUAL_REVIEW
r2 tobias-falk: MISSING_INFORMATION
r2 tobias-renner: MISSING_INFORMATION
completed 6, failed 0, still pending 0 (188s elapsed)


## 7. Evaluation

The framework agreed before any eval code was written.

1. **Accuracy against documented expectations.** `../../samples/blank-documents/eval-tuples.md`
   names expected application results for 9 personas, called core, and equates 5 more with core
   tuples, called derived. **Both** systems are scored against these labels, so the rules engine
   is audited too rather than assumed correct. The tuples document itself warns that the sample
   set "is not enough to call the rule engine correct".
2. **Agreement matrix.** The prototype's modal status across repeats against the rules engine,
   over all personas.
3. **Stability, meaning flip rate.** Does the prototype change its answer across repeat runs on
   identical input? The rules engine's flip rate is 0 by construction.
4. **Disagreement table.** Every mismatch with both rationales, for manual adjudication, written
   to `runs/disagreements.md`.
5. **Cost and latency.** From `runs/ledger.jsonl`. Reported, and not a headline metric.

Outputs are persisted to `runs/eval-summary.json` and `runs/comparison.csv`. Two personas are
unlabeled, which are `daniel-roth`, needing program-subject fact variants, and
`katharina-berger`, whose nursing overlay is outside the current policy.


In [8]:
import pandas as pd

# Documented expectations from ../../samples/blank-documents/eval-tuples.md.
# basis "core": an Existing tuple row names the persona and its expected application result.
# basis "derived": the doc's "existing variants" section equates the persona with a core
# tuple (school-part-only FHR repeats tuple 7; melina-sturm is a variant of tuple 18;
# oemer-yilmaz is a second complete FHR like tuple 5).
EXPECTED = {
    "felix-brandt":       ("ELIGIBLE", "core", "tuple 1"),
    "sofia-lorenz":       ("ELIGIBLE", "core", "tuple 2"),
    "erika-musterfrau":   ("ELIGIBLE", "core", "tuple 5"),
    "jonas-krause":       ("MISSING_INFORMATION", "core", "tuple 7"),
    "claudia-siebert":    ("ELIGIBLE", "core", "tuple 9"),
    "stefan-brenner":     ("MANUAL_REVIEW", "core", "tuple 12"),
    "katrin-vogel":       ("MANUAL_REVIEW", "core", "tuple 16"),
    "tobias-falk":        ("INELIGIBLE", "core", "tuple 17"),
    "tobias-renner":      ("MISSING_INFORMATION", "core", "tuple 18"),
    "anna-beispiel":      ("MISSING_INFORMATION", "derived", "repeats tuple 7"),
    "max-mustermann":     ("MISSING_INFORMATION", "derived", "repeats tuple 7"),
    "lena-schmidt-weber": ("MISSING_INFORMATION", "derived", "repeats tuple 7"),
    "melina-sturm":       ("MISSING_INFORMATION", "derived", "variant of tuple 18"),
    "oemer-yilmaz":       ("ELIGIBLE", "derived", "second complete FHR (tuple 5)"),
}


def load_baseline() -> dict:
    """Rule-based results regenerated by baseline.sh (fresh `admissions screen` runs)."""
    rows = {}
    if not BASELINE_DIR.exists():
        return rows
    for d in sorted(p for p in BASELINE_DIR.iterdir() if p.is_dir()):
        result = d / "application-result.json"
        failure = d / "processing-failure.json"
        if result.exists():
            j = json.loads(result.read_text())
            rows[d.name] = {
                "status": j["application_status"],
                "reason": j["application_reason_code"],
                "headline": j["summary"]["canonical"]["headline"],
                "missing": [m["label"] for m in j.get("missing_information", [])],
                "manual": [m["reason_code"] for m in j.get("manual_review", [])],
            }
        elif failure.exists():
            j = json.loads(failure.read_text())
            rows[d.name] = {"status": "RUN_FAILED", "reason": j.get("code", "UNKNOWN"),
                            "headline": j.get("safe_message", ""), "missing": [], "manual": []}
    return rows


proto = _load_records()
baseline = load_baseline()
print(f"prototype records: {len(proto)}   baseline personas: {len(baseline)}   labels: {len(EXPECTED)}")

prototype records: 48   baseline personas: 16   labels: 14


In [9]:
def modal(values: list[str]) -> str:
    return max(sorted(set(values)), key=values.count)


rows = []
for persona in PERSONAS:
    statuses = [proto[(persona, r)]["application_status"] for r in range(N_REPEATS) if (persona, r) in proto]
    b = baseline.get(persona, {})
    exp = EXPECTED.get(persona)
    rows.append({
        "persona": persona,
        "expected": exp[0] if exp else None,
        "basis": exp[1] if exp else None,
        "rules_status": b.get("status"),
        "rules_reason": b.get("reason"),
        "proto_runs": len(statuses),
        "proto_statuses": statuses,
        "proto_modal": modal(statuses) if statuses else None,
        "proto_stable": len(set(statuses)) == 1 if statuses else None,
    })
df = pd.DataFrame(rows)
df[["persona", "expected", "rules_status", "proto_modal", "proto_statuses", "proto_stable"]]

,persona,expected,rules_status,proto_modal,proto_statuses,proto_stable
0,anna-beispiel,MISSING_INFORMATION,INELIGIBLE,MISSING_INFORMATION,"[MISSING_INFORMATION, MISSING_INFORMATION, MIS...",True
1,claudia-siebert,ELIGIBLE,ELIGIBLE,MISSING_INFORMATION,"[MISSING_INFORMATION, MANUAL_REVIEW, MISSING_I...",False
2,daniel-roth,NaN,MISSING_INFORMATION,MISSING_INFORMATION,"[CONDITIONALLY_ELIGIBLE, MISSING_INFORMATION, ...",False
3,erika-musterfrau,ELIGIBLE,MANUAL_REVIEW,MANUAL_REVIEW,"[MANUAL_REVIEW, ELIGIBLE, MANUAL_REVIEW]",False
4,felix-brandt,ELIGIBLE,ELIGIBLE,ELIGIBLE,"[ELIGIBLE, MISSING_INFORMATION, ELIGIBLE]",False
5,jonas-krause,MISSING_INFORMATION,MISSING_INFORMATION,MISSING_INFORMATION,"[MISSING_INFORMATION, MISSING_INFORMATION, MIS...",True
6,katharina-berger,NaN,MISSING_INFORMATION,MISSING_INFORMATION,"[MISSING_INFORMATION, MISSING_INFORMATION, MIS...",True
7,katrin-vogel,MANUAL_REVIEW,MISSING_INFORMATION,MISSING_INFORMATION,"[MISSING_INFORMATION, MISSING_INFORMATION, MIS...",True
8,lena-schmidt-weber,MISSING_INFORMATION,INELIGIBLE,MISSING_INFORMATION,"[MISSING_INFORMATION, MISSING_INFORMATION, MIS...",True
9,max-mustermann,MISSING_INFORMATION,MISSING_INFORMATION,MISSING_INFORMATION,"[MISSING_INFORMATION, MISSING_INFORMATION, MIS...",True


In [10]:
labeled = df[df.expected.notna() & df.proto_modal.notna() & df.rules_status.notna()]
core = labeled[labeled.basis == "core"]
both = df[df.proto_modal.notna() & df.rules_status.notna()]
multi = df[df.proto_runs >= 2]


def acc(frame: pd.DataFrame, col: str) -> float | None:
    return round(float((frame[col] == frame.expected).mean()), 3) if len(frame) else None


summary = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model": MODEL,
    "critic_enabled": ENABLE_CRITIC,
    "n_personas": int(len(df)),
    "prototype_runs": int(df.proto_runs.sum()),
    "accuracy_vs_labels": {
        "n_core": int(len(core)),
        "rules_core": acc(core, "rules_status"),
        "prototype_core": acc(core, "proto_modal"),
        "n_all_labeled": int(len(labeled)),
        "rules_all_labeled": acc(labeled, "rules_status"),
        "prototype_all_labeled": acc(labeled, "proto_modal"),
    },
    "agreement_prototype_vs_rules": {
        "n": int(len(both)),
        "rate": round(float((both.proto_modal == both.rules_status).mean()), 3) if len(both) else None,
    },
    "stability": {
        "n_personas_with_repeats": int(len(multi)),
        "personas_with_flips": int((~multi.proto_stable.astype(bool)).sum()) if len(multi) else None,
        "flip_rate": round(float((~multi.proto_stable.astype(bool)).mean()), 3) if len(multi) else None,
        "note": "rules engine flip-rate is 0 by construction (deterministic evaluator)",
    },
}
print(json.dumps(summary, indent=2))

if len(both):
    print("\nAgreement matrix (rows=prototype modal, cols=rules engine):")
    print(pd.crosstab(both.proto_modal, both.rules_status))

(RUNS_DIR / "eval-summary.json").write_text(json.dumps(summary, indent=2))
df.assign(proto_statuses=df.proto_statuses.apply(lambda s: "|".join(s))).to_csv(RUNS_DIR / "comparison.csv", index=False)
print(f"\nwrote {RUNS_DIR / 'eval-summary.json'} and {RUNS_DIR / 'comparison.csv'}")

{
  "generated_at": "2026-08-25T20:57:35.427569+00:00",
  "model": "gpt-5.6-terra",
  "critic_enabled": false,
  "n_personas": 16,
  "prototype_runs": 48,
  "accuracy_vs_labels": {
    "n_core": 9,
    "rules_core": 0.667,
    "prototype_core": 0.444,
    "n_all_labeled": 14,
    "rules_all_labeled": 0.571,
    "prototype_all_labeled": 0.571
  },
  "agreement_prototype_vs_rules": {
    "n": 16,
    "rate": 0.688
  },
  "stability": {
    "n_personas_with_repeats": 16,
    "personas_with_flips": 7,
    "flip_rate": 0.438,
    "note": "rules engine flip-rate is 0 by construction (deterministic evaluator)"
  }
}

Agreement matrix (rows=prototype modal, cols=rules engine):
rules_status         ELIGIBLE  INELIGIBLE  MANUAL_REVIEW  MISSING_INFORMATION
proto_modal                                                                  
ELIGIBLE                    2           0              0                    0
MANUAL_REVIEW               0           0              1                    0
MISSING_IN

In [11]:
# Disagreement table for manual adjudication: any persona where expected, rules engine,
# and prototype (modal) do not all coincide.
lines = ["# Disagreements — manual adjudication", "",
         f"Generated {datetime.now(timezone.utc).isoformat()} · model {MODEL} · critic={ENABLE_CRITIC}", ""]
n_disagreements = 0
for _, row in df.iterrows():
    if row.proto_modal is None or row.rules_status is None:
        continue
    verdicts = {v for v in [row.expected, row.rules_status, row.proto_modal] if v is not None}
    if len(verdicts) <= 1:
        continue
    n_disagreements += 1
    b = baseline.get(row.persona, {})
    latest = next(proto[(row.persona, r)] for r in reversed(range(N_REPEATS)) if (row.persona, r) in proto)
    d = latest["decision"]
    lines += [
        f"## {row.persona}",
        "",
        f"- expected: **{row.expected or '(unlabeled)'}** ({row.basis or '-'})",
        f"- rules engine: **{row.rules_status}** (`{row.rules_reason}`) — {b.get('headline', '')}",
        f"  - missing: {b.get('missing', [])}  manual: {b.get('manual', [])}",
        f"- prototype: **{row.proto_modal}** across repeats {row.proto_statuses}",
        f"  - rationale (latest run): {d['rationale']}",
        f"  - missing: {d['missing_information']}  manual: {d['manual_review_reasons']}  conditions: {d['conditions']}",
        f"  - adjudication: _(fill in: which system is right, and why)_",
        "",
    ]
if n_disagreements == 0:
    lines.append("No disagreements with the available data.")
(RUNS_DIR / "disagreements.md").write_text("\n".join(lines))
print(f"{n_disagreements} disagreement(s) -> {RUNS_DIR / 'disagreements.md'}")
for l in lines[:60]:
    print(l)

10 disagreement(s) -> experiment/admissions-full-agentic-prototype/runs/disagreements.md
# Disagreements — manual adjudication

Generated 2026-08-25T20:57:35.443614+00:00 · model gpt-5.6-terra · critic=False

## anna-beispiel

- expected: **MISSING_INFORMATION** (derived)
- rules engine: **INELIGIBLE** (`ACADEMIC_ACCESS_INELIGIBLE`) — Academic access requirements are not satisfied
  - missing: []  manual: []
- prototype: **MISSING_INFORMATION** across repeats ['MISSING_INFORMATION', 'MISSING_INFORMATION', 'MISSING_INFORMATION']
  - rationale (latest run): The submitted document certifies only the school-based part of a German Fachhochschulreife. It does not establish the required vocational/practical part, so it does not yet prove a complete Fachhochschulreife that grants direct access to the B.Sc. Computer Science programme.
  - missing: ['Evidence of the vocational/practical part of the Fachhochschulreife (or a combined overall Fachhochschulreife certificate).', 'If the practical par

In [12]:
# Cost & latency from the ledger (secondary metric).
led = pd.DataFrame(json.loads(l) for l in LEDGER_PATH.open())
by_node = led.groupby("node").agg(
    calls=("node", "size"),
    mean_input_tokens=("input_tokens", "mean"),
    mean_output_tokens=("output_tokens", "mean"),
    mean_duration_s=("duration_ms", lambda s: s.mean() / 1000),
    total_input_tokens=("input_tokens", "sum"),
    total_output_tokens=("output_tokens", "sum"),
).round(1)
print(by_node)
print(f"\ntotal: {led.input_tokens.sum():,.0f} input / {led.output_tokens.sum():,.0f} output tokens "
      f"across {len(led)} calls")
by_node.to_json(RUNS_DIR / "cost-summary.json", indent=2)
print(f"wrote {RUNS_DIR / 'cost-summary.json'}")

                calls  mean_input_tokens  mean_output_tokens  mean_duration_s  \
node                                                                            
decision_maker     48            15206.3              1545.0             20.9   
policy_analyst     48            93542.0              5988.1             69.2   

                total_input_tokens  total_output_tokens  
node                                                     
decision_maker              729902                74161  
policy_analyst             4490016               287427  

total: 5,219,918 input / 361,588 output tokens across 96 calls
wrote experiment/admissions-full-agentic-prototype/runs/cost-summary.json
